# YOLOE → Any6D Pipeline

This notebook connects YOLOE (open-vocabulary detection) with Any6D (6D pose estimation).

**Flow:**
```
Anchor image + Scene image
        ↓
   YOLOE (visual prompt)
        ↓
   bbox + mask (scene)
        ↓
   Any6D (FoundationPose)
        ↓
   6D Pose (4×4 matrix)
```

**Run this notebook inside the Any6D Docker container:**
```bash
docker compose run --rm -p 8888:8888 any6d jupyter lab --ip=0.0.0.0 --no-browser
```

## Cell 1 — Imports & Setup

In [ ]:
import os
import sys
import cv2
import numpy as np
import torch
import trimesh
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# ── Any6D path setup ──────────────────────────────────────────
WORKSPACE = '/workspace'
sys.path.insert(0, WORKSPACE)
sys.path.insert(0, f'{WORKSPACE}/foundationpose')
sys.path.insert(0, f'{WORKSPACE}/foundationpose/mycpp/build')

# ── YOLOE (Ultralytics) ───────────────────────────────────────
from ultralytics import YOLOE
from ultralytics.models.yolo.yoloe import YOLOEVPSegPredictor

# ── Any6D ─────────────────────────────────────────────────────
import nvdiffrast.torch as dr
from foundationpose.Utils import *
from estimater import Any6D

print('✅ All imports OK')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0)}')

## Cell 2 — Load Images

In [ ]:
# ── Paths — adjust to your images ────────────────────────────
ANCHOR_IMAGE_PATH = '/workspace/images/anchor.jpg'   # reference image of the object
SCENE_IMAGE_PATH  = '/workspace/images/scene.jpg'    # scene where object is
DEPTH_IMAGE_PATH  = '/workspace/images/depth.png'    # depth image (16-bit)
MESH_PATH         = '/workspace/demo_data/mustard.obj'  # 3D mesh of object

# Camera intrinsics (adjust to your camera)
fx, fy = 614.0, 614.0
cx, cy = 320.0, 240.0
K = np.array([[fx, 0, cx],
              [0, fy, cy],
              [0,  0,  1]], dtype=np.float64)

depth_scale = 1000.0  # mm → meters

# ── Load images ───────────────────────────────────────────────
anchor_bgr = cv2.imread(ANCHOR_IMAGE_PATH)
scene_bgr  = cv2.imread(SCENE_IMAGE_PATH)
anchor_rgb = cv2.cvtColor(anchor_bgr, cv2.COLOR_BGR2RGB)
scene_rgb  = cv2.cvtColor(scene_bgr,  cv2.COLOR_BGR2RGB)

depth = cv2.imread(DEPTH_IMAGE_PATH, cv2.IMREAD_ANYDEPTH).astype(np.float32) / depth_scale

# ── Display ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(anchor_rgb); axes[0].set_title('Anchor (reference)')
axes[1].imshow(scene_rgb);  axes[1].set_title('Scene')
axes[2].imshow(depth, cmap='jet'); axes[2].set_title('Depth')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.show()

print(f'Anchor: {anchor_rgb.shape}')
print(f'Scene:  {scene_rgb.shape}')
print(f'Depth:  {depth.shape}, range: [{depth.min():.3f}, {depth.max():.3f}] m')

## Cell 3 — YOLOE: Detect object in Scene using Anchor as visual prompt

In [ ]:
# ── Load YOLOE model ──────────────────────────────────────────
yoloe_model = YOLOE('yoloe-11l-seg.pt')  # adjust model size if needed
yoloe_model.to('cuda')

# ── Get bbox of object in anchor image ────────────────────────
# You can set this manually or use a simple detector
# Format: [x1, y1, x2, y2] in anchor image coordinates
h_a, w_a = anchor_rgb.shape[:2]

# Option A: Use full anchor image as bbox
anchor_bbox = np.array([[0, 0, w_a, h_a]])

# Option B: Set manually if you know the object location
# anchor_bbox = np.array([[100, 50, 400, 380]])  # [x1, y1, x2, y2]

# ── Visual prompt ─────────────────────────────────────────────
visual_prompts = {
    'cls': [0],
    'bboxes': anchor_bbox
}

# ── Run YOLOE on scene using anchor as visual prompt ──────────
print('Running YOLOE...')
results = yoloe_model.predict(
    SCENE_IMAGE_PATH,
    refer_image=ANCHOR_IMAGE_PATH,
    visual_prompts=visual_prompts,
    predictor=YOLOEVPSegPredictor
)

result = results[0]
print(f'Detections: {len(result.boxes)}')

# ── Extract best detection ────────────────────────────────────
if len(result.boxes) == 0:
    raise ValueError('No object detected! Check your anchor image and bbox.')

# Take highest confidence detection
best_idx = result.boxes.conf.argmax().item()
bbox_xyxy = result.boxes.xyxy[best_idx].cpu().numpy().astype(int)
confidence = result.boxes.conf[best_idx].item()

print(f'Best detection: bbox={bbox_xyxy}, conf={confidence:.3f}')

# ── Extract mask ──────────────────────────────────────────────
if result.masks is not None:
    mask = result.masks.data[best_idx].cpu().numpy()
    # Resize mask to scene image size
    mask = cv2.resize(mask, (scene_rgb.shape[1], scene_rgb.shape[0]))
    mask_bool = mask > 0.5
    print(f'Mask: {mask_bool.shape}, pixels={mask_bool.sum()}')
else:
    # Fallback: create mask from bbox
    mask_bool = np.zeros(scene_rgb.shape[:2], dtype=bool)
    x1, y1, x2, y2 = bbox_xyxy
    mask_bool[y1:y2, x1:x2] = True
    print('No segmentation mask — using bbox mask as fallback')

# ── Visualize YOLOE result ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Scene with bbox
axes[0].imshow(scene_rgb)
x1, y1, x2, y2 = bbox_xyxy
rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                          linewidth=3, edgecolor='lime', facecolor='none')
axes[0].add_patch(rect)
axes[0].set_title(f'YOLOE Detection (conf={confidence:.2f})')
axes[0].axis('off')

# Mask overlay
overlay = scene_rgb.copy()
overlay[mask_bool] = overlay[mask_bool] * 0.5 + np.array([0, 255, 0]) * 0.5
axes[1].imshow(overlay.astype(np.uint8))
axes[1].set_title('Segmentation Mask')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## Cell 4 — Any6D: 6D Pose Estimation using YOLOE mask

In [ ]:
# ── Load 3D mesh ──────────────────────────────────────────────
print('Loading mesh...')
mesh = trimesh.load(MESH_PATH)
print(f'Mesh: {len(mesh.vertices)} vertices, {len(mesh.faces)} faces')

# ── Initialize Any6D ──────────────────────────────────────────
print('Initializing Any6D...')
glctx = dr.RasterizeCudaContext()
save_path = '/workspace/results/yoloe_any6d'
os.makedirs(save_path, exist_ok=True)

estimator = Any6D(
    symmetry_tfs=None,
    mesh=mesh,
    debug_dir=save_path,
    debug=2
)
print('✅ Any6D initialized')

## Cell 5 — Run Pose Estimation

In [ ]:
# ── Run Any6D with YOLOE mask ─────────────────────────────────
print('Running Any6D pose estimation...')
print(f'  Input: RGB {scene_rgb.shape}, Depth {depth.shape}')
print(f'  Mask pixels: {mask_bool.sum()}')

pose = estimator.register_any6d(
    K=K,
    rgb=scene_rgb,
    depth=depth,
    ob_mask=mask_bool,
    iteration=5,
    name='yoloe_any6d'
)

print('\n✅ Pose estimation complete!')
print('\n4×4 Pose matrix (camera → object):')
print(pose)

# ── Interpret pose ────────────────────────────────────────────
R = pose[:3, :3]  # rotation matrix
t = pose[:3, 3]   # translation vector (meters)

print(f'\nObject position:')
print(f'  X = {t[0]*100:.1f} cm')
print(f'  Y = {t[1]*100:.1f} cm')
print(f'  Z = {t[2]*100:.1f} cm (depth)')

# ── Save pose ─────────────────────────────────────────────────
np.savetxt(os.path.join(save_path, 'pose.txt'), pose)
print(f'\nPose saved to {save_path}/pose.txt')

## Cell 6 — Visualize Result

In [ ]:
# ── Project mesh onto scene using estimated pose ──────────────
def project_mesh_corners(mesh, pose, K, img):
    """Project 3D mesh bounding box corners onto 2D image."""
    bounds = mesh.bounds  # [[xmin,ymin,zmin], [xmax,ymax,zmax]]
    corners_3d = np.array([
        [bounds[0,0], bounds[0,1], bounds[0,2]],
        [bounds[1,0], bounds[0,1], bounds[0,2]],
        [bounds[1,0], bounds[1,1], bounds[0,2]],
        [bounds[0,0], bounds[1,1], bounds[0,2]],
        [bounds[0,0], bounds[0,1], bounds[1,2]],
        [bounds[1,0], bounds[0,1], bounds[1,2]],
        [bounds[1,0], bounds[1,1], bounds[1,2]],
        [bounds[0,0], bounds[1,1], bounds[1,2]],
    ])
    
    # Transform to camera frame
    corners_cam = (pose[:3,:3] @ corners_3d.T + pose[:3,3:]).T
    
    # Project to 2D
    corners_2d = (K @ corners_cam.T).T
    corners_2d = corners_2d[:, :2] / corners_2d[:, 2:]
    
    return corners_2d.astype(int)

def draw_bbox_3d(img, corners_2d, color=(0, 255, 0), thickness=2):
    """Draw 3D bounding box on image."""
    img = img.copy()
    edges = [(0,1),(1,2),(2,3),(3,0),  # bottom
             (4,5),(5,6),(6,7),(7,4),  # top
             (0,4),(1,5),(2,6),(3,7)]  # sides
    for i, j in edges:
        p1 = tuple(corners_2d[i])
        p2 = tuple(corners_2d[j])
        cv2.line(img, p1, p2, color, thickness)
    return img

# ── Visualize ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# 1. Anchor
axes[0].imshow(anchor_rgb)
axes[0].set_title('Anchor (reference)')
axes[0].axis('off')

# 2. YOLOE detection
axes[1].imshow(overlay.astype(np.uint8))
axes[1].set_title(f'YOLOE Detection (conf={confidence:.2f})')
axes[1].axis('off')

# 3. Any6D pose
try:
    corners_2d = project_mesh_corners(mesh, pose, K, scene_rgb)
    result_img = draw_bbox_3d(scene_rgb, corners_2d, color=(0, 255, 0))
    # Add mask overlay
    result_img[mask_bool] = result_img[mask_bool] * 0.7 + np.array([0, 100, 255]) * 0.3
    axes[2].imshow(result_img.astype(np.uint8))
    axes[2].set_title('6D Pose (Any6D)')
except Exception as e:
    axes[2].imshow(scene_rgb)
    axes[2].set_title(f'6D Pose (visualization error: {e})')
axes[2].axis('off')

plt.suptitle('YOLOE → Any6D Pipeline Result', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(save_path, 'pipeline_result.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'Result saved to {save_path}/pipeline_result.png')

## Cell 7 — Full Pipeline Class (reusable)

In [ ]:
class YOLOE_Any6D_Pipeline:
    """
    Full pipeline: Anchor image → YOLOE detection → Any6D 6D pose estimation.
    
    Usage:
        pipeline = YOLOE_Any6D_Pipeline(mesh_path='object.obj', K=camera_K)
        pose = pipeline.run(
            anchor_path='anchor.jpg',
            scene_path='scene.jpg',
            depth_path='depth.png',
            anchor_bbox=[x1, y1, x2, y2]  # optional
        )
    """
    
    def __init__(
        self,
        mesh_path,
        K,
        yoloe_model='yoloe-11l-seg.pt',
        depth_scale=1000.0,
        save_dir='/workspace/results',
        conf_threshold=0.25
    ):
        self.K = K
        self.depth_scale = depth_scale
        self.save_dir = save_dir
        self.conf_threshold = conf_threshold
        os.makedirs(save_dir, exist_ok=True)
        
        print('[1/3] Loading YOLOE...')
        self.yoloe = YOLOE(yoloe_model)
        self.yoloe.to('cuda')
        
        print('[2/3] Loading mesh...')
        self.mesh = trimesh.load(mesh_path)
        
        print('[3/3] Initializing Any6D...')
        self.glctx = dr.RasterizeCudaContext()
        self.estimator = Any6D(
            symmetry_tfs=None,
            mesh=self.mesh,
            debug_dir=save_dir,
            debug=0
        )
        print('✅ Pipeline ready!')
    
    def detect(self, anchor_path, scene_path, anchor_bbox=None):
        """Stage 1: YOLOE detection using anchor as visual prompt."""
        anchor = cv2.imread(anchor_path)
        h, w = anchor.shape[:2]
        
        bbox = np.array([anchor_bbox]) if anchor_bbox else np.array([[0, 0, w, h]])
        
        results = self.yoloe.predict(
            scene_path,
            refer_image=anchor_path,
            visual_prompts={'cls': [0], 'bboxes': bbox},
            predictor=YOLOEVPSegPredictor
        )
        
        result = results[0]
        if len(result.boxes) == 0:
            raise ValueError('No object detected!')
        
        # Best detection
        idx = result.boxes.conf.argmax().item()
        bbox_xyxy = result.boxes.xyxy[idx].cpu().numpy().astype(int)
        conf = result.boxes.conf[idx].item()
        
        scene = cv2.imread(scene_path)
        h_s, w_s = scene.shape[:2]
        
        if result.masks is not None:
            mask = result.masks.data[idx].cpu().numpy()
            mask = cv2.resize(mask, (w_s, h_s)) > 0.5
        else:
            mask = np.zeros((h_s, w_s), dtype=bool)
            x1, y1, x2, y2 = bbox_xyxy
            mask[y1:y2, x1:x2] = True
        
        print(f'  → Detected with conf={conf:.3f}, mask pixels={mask.sum()}')
        return bbox_xyxy, mask, conf
    
    def estimate_pose(self, scene_path, depth_path, mask, name='run'):
        """Stage 2: Any6D 6D pose estimation."""
        scene_bgr = cv2.imread(scene_path)
        scene_rgb = cv2.cvtColor(scene_bgr, cv2.COLOR_BGR2RGB)
        depth = cv2.imread(depth_path, cv2.IMREAD_ANYDEPTH).astype(np.float32) / self.depth_scale
        
        pose = self.estimator.register_any6d(
            K=self.K,
            rgb=scene_rgb,
            depth=depth,
            ob_mask=mask,
            iteration=5,
            name=name
        )
        return pose
    
    def run(self, anchor_path, scene_path, depth_path, anchor_bbox=None, name='run'):
        """Run the full pipeline."""
        print(f'\n{"="*50}')
        print('YOLOE → Any6D Pipeline')
        print(f'{"="*50}')
        
        print('\n[Stage 1] YOLOE Detection...')
        bbox, mask, conf = self.detect(anchor_path, scene_path, anchor_bbox)
        
        print('\n[Stage 2] Any6D Pose Estimation...')
        pose = self.estimate_pose(scene_path, depth_path, mask, name=name)
        
        t = pose[:3, 3]
        print(f'\n✅ Done!')
        print(f'Object position: X={t[0]*100:.1f}cm, Y={t[1]*100:.1f}cm, Z={t[2]*100:.1f}cm')
        
        # Save
        np.savetxt(os.path.join(self.save_dir, f'{name}_pose.txt'), pose)
        
        return pose, bbox, mask


# ── Example usage ──────────────────────────────────────────────
K = np.array([[614, 0, 320],
              [0, 614, 240],
              [0,   0,   1]], dtype=np.float64)

pipeline = YOLOE_Any6D_Pipeline(
    mesh_path='/workspace/demo_data/mustard.obj',
    K=K,
    save_dir='/workspace/results/pipeline'
)

pose, bbox, mask = pipeline.run(
    anchor_path='/workspace/images/anchor.jpg',
    scene_path='/workspace/images/scene.jpg',
    depth_path='/workspace/images/depth.png',
    name='demo'
)

print('\nFinal pose:')
print(pose)